## Celda 1: Importaciones

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error
from misr_advanced import MISR_Model
from nuclearpy_models.models.BE.sr_fast import sr_fast_be


## Celda 2: Carga de Datos
#### Cargamos los datasets de entrenamiento y prueba.

In [ ]:
# 1. Cargar el dataset
path_train = "Data/Experimental/be_train.csv"
path_test = "Data/Experimental/be_test.csv"

train_df = pd.read_csv(path_train)
test_df = pd.read_csv(path_test)

# 2. Preparar las muestras (usando los dataframes completos)
sample_train = train_df
sample_test = test_df

print(f"Datasets cargados. Entrenamiento: {len(sample_train)} núcleos, Test: {len(sample_test)} núcleos.")


## Celda 3: Configuración del Modelo
#### Definimos los parámetros para el modelo MISR (generaciones, población, etc.).

In [ ]:
# 3. Configurar el modelo con parámetros balanceados
model = MISR_Model(
    maxiter=10,           
    k_folds=5,           
    population_size=1000, 
    n_generations=15,
    s_features=4  
)


## Celda 4: Entrenamiento y Guardado
#### Ejecutamos el proceso de ajuste (fit) y guardamos el resultado en un archivo .pkl.

In [ ]:
# 4. Ejecutar el entrenamiento
print("Entrenando nuevo modelo MISR con sets explícitos...")
model.fit(sample_train, sample_test)

# Guardar el modelo entrenado
joblib.dump(model, 'misr_model.pkl')
print("Modelo guardado exitosamente en 'misr_model.pkl'")


## Celda 5: Generación de Predicciones y Comparativa
#### Generamos predicciones con el modelo recién entrenado y también con el modelo original del paper para comparar resultados.




In [ ]:
# 5. Preparar datos de comparación
print("Generando comparativa con el modelo original del paper...")
Y_true = model.Y_test
X_test_features = model.X_test
Extras_test = model.Extras_test # Contiene [Z, N] para el modelo original

# Predicciones del nuevo modelo
Y_pred_new = model.predict(pd.DataFrame(X_test_features, columns=model.feature_names))

# Predicciones del modelo del paper (SR Fast)
Y_pred_paper = []
for i in range(len(Extras_test)):
    Z, N = Extras_test[i]
    pred, _ = sr_fast_be(Z, N)
    Y_pred_paper.append(pred)
Y_pred_paper = np.array(Y_pred_paper)


## Celda 6: Cálculo de Métricas y Resultados Finales
#### Finalmente, calculamos el RMSE, MAE y R² para ambos modelos y mostramos la fórmula analítica encontrada por el nuevo modelo.

In [ ]:
# 6. Definir y calcular métricas
def get_metrics(true, pred):
    rmse = np.sqrt(mean_squared_error(true, pred))
    mae = mean_absolute_error(true, pred)
    return rmse, mae

rmse_new, mae_new = get_metrics(Y_true, Y_pred_new)
rmse_paper, mae_paper = get_metrics(Y_true, Y_pred_paper)

# Mostrar tabla de resultados
print("\n" + "="*40)
print(f"{'Métrica':<15} | {'Nuevo MISR':<12} | {'Paper SR'}")
print("-" * 40)
print(f"{'RMSE':<15} | {rmse_new:<12.4f} | {rmse_paper:.4f}")
print(f"{'MAE':<15} | {mae_new:<12.4f} | {mae_paper:.4f}")
print("="*40)

# Imprimir la expresión analítica final
print("\nModelo MISR - Expresión Analítica Final:")
print("-" * 40)
print(model.get_formula())
print("-" * 40)

# Calcular R² adicionales
residuals_new   = Y_true - Y_pred_new
residuals_paper = Y_true - Y_pred_paper
r2_new   = 1 - np.sum(residuals_new**2)   / np.sum((Y_true - Y_true.mean())**2)
r2_paper = 1 - np.sum(residuals_paper**2) / np.sum((Y_true - Y_true.mean())**2)

print(f"\nR²  MISR Nuevo : {r2_new:.4f}")
print(f"R²  Paper SR  : {r2_paper:.4f}")
